# Merge S5 — Train model 8-class CUỐI (Dataset020_KneeUnion)

Đây là model báo cáo — nơi **vắt sụn tối đa**: ResEnc + nhiều epoch. Dataset020 = 544 ca (404 ZIB + 140 iMorph), nhãn đầy đủ 8-class (GT + pseudo).

Cấu hình nâng sụn (đã bàn): **ResEnc**, epoch nhiều, batch dùng VRAM. Sau train → **S6** đánh giá + surface metrics.


In [ ]:
!pip install -q nnunetv2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 0) Env


In [ ]:
import os
os.environ["nnUNet_raw"]          = "/content/drive/MyDrive/nnUNet_raw"      # Dataset020 o day
os.environ["nnUNet_preprocessed"] = "/content/nnUNet_preprocessed"          # local nhanh
os.environ["nnUNet_results"]      = "/content/drive/MyDrive/nnUNet_results"  # Drive persist
os.makedirs(os.environ["nnUNet_preprocessed"], exist_ok=True)
print("raw datasets:", [d for d in os.listdir(os.environ["nnUNet_raw"]) if d.startswith("Dataset")])


## 1) plan_and_preprocess (ResEnc-L)

Dataset020 trộn ảnh ZIB + iMorph (cùng domain OAI DESS, có thể khác orientation) → nnUNet tự resample về spacing chung. Nhãn đã khớp geometry ảnh (S4 dùng CopyInformation) nên verify sẽ pass.


In [ ]:
!nnUNetv2_plan_and_preprocess -d 20 -pl nnUNetPlannerResEncL --verify_dataset_integrity


## 2) (Tùy chọn) tăng batch_size dùng VRAM

Chỉ tăng nếu muốn xài VRAM (KHÔNG tự nhanh hơn — 250 iter/epoch cố định). A100 40GB → batch 3; 80GB → 4–6.


In [ ]:
import json
p = "/content/nnUNet_preprocessed/Dataset020_KneeUnion/nnUNetResEncUNetLPlans.json"
j = json.load(open(p))
j["configurations"]["3d_fullres"]["batch_size"] = 3   # sua tuy VRAM; bo cell nay neu giu mac dinh
json.dump(j, open(p, "w"), indent=4)
print("batch_size ->", j["configurations"]["3d_fullres"]["batch_size"])


## 3) Train model 8-class (fold 0)

Chọn số epoch theo quỹ giờ Colab (~3 phút/epoch ResEnc-L, batch 2):
- `_250epochs` ≈ ~12h (1–2 session) — **đủ tốt, khuyên bắt đầu**.
- `_500epochs` ≈ ~25h — vắt thêm chút sụn, cần resume nhiều session.
- `_1000epochs` ≈ ~2 ngày — lý tưởng nhưng nặng cho Colab.

Colab đứt → chạy lại **đúng lệnh + `--c`** để resume (checkpoint ở Drive).


In [ ]:
!nnUNetv2_train 20 3d_fullres 0 -p nnUNetResEncUNetLPlans -tr nnUNetTrainer_250epochs


## 4) Theo dõi & bước sau
- `Pseudo dice [8 gia tri]` = [fem_bone, fem_cart, tib_bone, med_tib, lat_tib, **med_men**, **lat_men**, patellar].
- Mục tiêu: **sụn (2,4,5) vượt baseline** (0.881 / 0.835 / 0.855); **meniscus (6,7)** ~0.75–0.86.
- Sau khi train xong (hoặc plateau) → **S6**: đánh giá trên 3 held-out (ZIB Ts / iMorph test / SKM-TEA) + **surface metrics** (ASSD/HD95/thickness) cho sụn.
- Muốn model báo cáo mạnh hơn: train thêm fold 1–4 (5-fold ensemble) — làm sau khi fold 0 ổn.

Nếu muốn dùng hết VRAM cho **chất lượng** (không chỉ batch): thay bằng **ResEnc-XL**
(`-pl nnUNetPlannerResEncXL` ở cell 1, rồi `-p nnUNetResEncUNetXLPlans` ở cell 3) — patch lớn hơn, tốt cho sụn, nhưng nặng hơn.
